In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "robust_auditing").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from robust_auditing.mt_bench.results import (
    CATEGORIES,
    MODEL_LABELS,
    build_lineage_plot_rows,
    load_single_judgments,
    model_category_scores,
    model_scalar_scores,
)

JUDGMENT_FILE = ROOT / "artifacts" / "mt_bench" / "model_judgment" / "gpt-4_single.jsonl"


In [ ]:
df = load_single_judgments(JUDGMENT_FILE)
category_scores = model_category_scores(df).reset_index()
scalar_scores = model_scalar_scores(df).reset_index()

category_scores["label"] = category_scores["model"].map(MODEL_LABELS).fillna(category_scores["model"])
scalar_scores["label"] = scalar_scores["model"].map(MODEL_LABELS).fillna(scalar_scores["model"])
scalar_scores.sort_values("score", ascending=False)


In [ ]:
radar_models = [
    "olmo2_1b_sft",
    "olmo2_1b_dpo",
    "olmo2_1b_rlvr1",
    "olmo2_1b_instruct",
    "grpo_10k_ft_leftpad",
    "passed_final_poisoning_ft_balanced115_seed3",
]
radar_df = category_scores[category_scores["model"].isin(radar_models)].copy()
radar_df["model"] = pd.Categorical(radar_df["model"], categories=radar_models, ordered=True)
radar_df = radar_df.sort_values(["model", "category"])

fig_radar = px.line_polar(
    radar_df,
    r="score",
    theta="category",
    line_close=True,
    category_orders={"category": CATEGORIES},
    color="label",
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_radar.update_layout(font={"size": 16})
fig_radar.show()


In [ ]:
score_lookup = dict(zip(scalar_scores["model"], scalar_scores["score"]))
lineage_rows = build_lineage_plot_rows(score_lookup)
lineage_df = pd.DataFrame(lineage_rows)

fig_lineage = go.Figure()
for branch, branch_df in lineage_df.groupby("branch", sort=False):
    branch_df = branch_df.sort_values("x")
    fig_lineage.add_trace(
        go.Scatter(
            x=branch_df["stage"],
            y=branch_df["score"],
            mode="lines+markers",
            name=branch,
            marker={"symbol": branch_df["marker"].iloc[-1], "size": 10},
            line={"color": branch_df["color"].iloc[-1], "width": 3},
        )
    )

fig_lineage.update_layout(
    xaxis_title="Checkpoint",
    yaxis_title="Mean MT-Bench score",
    xaxis={"categoryorder": "array", "categoryarray": ["SFT", "DPO", "RLVR1", "Instruct", "Fine-Tuned Instruct"]},
    font={"size": 16},
)
fig_lineage.show()


In [ ]:
fig_radar.write_image(ROOT / "artifacts" / "mt_bench" / "mt_bench_radar.png", width=900, height=700, scale=2)
fig_lineage.write_image(ROOT / "artifacts" / "mt_bench" / "mt_bench_lineage.png", width=900, height=500, scale=2)
